In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import os
import sys

%matplotlib inline

from _helpers import mock_snakemake
snakemake = mock_snakemake(
    "plot_mga",
)

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
epsilon = np.array([0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0])

plots = [
    dict(
        title="Instable trade routes (chokepoints)",
        upper=np.array([20, 22.5, 24.5, 26.0, 27.2, 28.1, 28.8, 29.4, 29.8]),
        lower=np.array([20, 17.5, 15.5, 14.0, 12.8, 11.9, 11.2, 10.6, 10.2]),
        y_max=200,
        y_mid=20,           # curves originate at 20
        color="#185FA5",
        alpha=0.25,
        arrow_up=None,
        arrow_dn=["Decreased", "high-risk routes"],
        y_label_add="through trade chokepoints",
        show_upper=False,
        show_lower=True,
        # Regional decomposition for lower curve
        regions=[
            dict(name="Malacca", values=np.array([8, 7, 6.5, 6.0, 5.5, 5.2, 4.8, 4.5, 4.2]), color="#1f77b4", linestyle="-", alpha=0.6),
            dict(name="Hormuz", values=np.array([8, 8, 8, 8, 8, 8, 8, 8, 8]), color="#ff7f0e", linestyle="-", alpha=0.6),
            dict(name="Panama", values=np.array([4, 2.5, 1.5, 0.0, -0.7, -1.2, -1.6, -1.9, -2.2]), color="#2ca02c", linestyle="--", alpha=0.6),
        ]
    ),
    dict(
        title="Stable exporters",
        upper=np.array([100, 118, 133, 147, 158, 167, 175, 182, 188]),
        lower=np.array([100,  82,  67,  54,  43,  34,  27,  21,  16]),
        y_max=200,
        y_mid=100,
        color="#3B6D11",
        alpha=0.25,
        arrow_up=["Increased stable", "HBI producers"],
        arrow_dn=None,
        y_label_add="in stable regions",
        show_upper=True,
        show_lower=False,
        # Regional decomposition for upper curve
        regions=[
            dict(name="South America", values=np.array([50, 62, 75, 87, 97, 106, 113, 119, 124]), color="#d62728", linestyle="-", alpha=0.6),
            dict(name="Oceania", values=np.array([50, 56, 58, 60, 61, 61, 62, 63, 64]), color="#9467bd", linestyle="-", alpha=0.6),
            dict(name="North America", values=np.array([20, 15, 12, 10, 8, 6, 5, 4, 3]), color="#8c564b", linestyle="--", alpha=0.6),
            dict(name="Europe", values=np.array([15, 12, 10, 8, 6, 4, 3, 2, 1]), color="#e377c2", linestyle="--", alpha=0.6),
            dict(name="Asia", values=np.array([25, 20, 15, 10, 8, 5, 3, 2, 1]), color="#bcbd22", linestyle="--", alpha=0.6),
        ]
    ),
    dict(
        title="Fragmentation/Block trade",
        upper=np.array([37.5, 44, 50, 55, 59, 63, 66, 69, 71]),
        lower=np.array([37.5, 31, 25, 20, 16, 13, 10,  8,  6]),
        y_max=200,
        y_mid=37.5,
        color="#534AB7",
        alpha=0.25,
        arrow_up=None,
        arrow_dn=["Decreased trade", "between blocks"],
        y_label_add="between Block A and B",
        show_upper=False,
        show_lower=True,
        # Regional decomposition for lower curve
        regions=[
            dict(name="China", values=np.array([20, 17, 14, 11, 9, 7, 5, 4, 3]), color="#17becf", linestyle="-", alpha=0.6),
            dict(name="Oceania", values=np.array([17.5, 14, 11, 9, 7, 6, 5, 4, 3]), color="#aec7e8", linestyle="-", alpha=0.6),
        ]
    ),
]

# ── Layout ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.subplots_adjust(wspace=0.35)

ARROW_X   = 4.5          # x position of arrows (data coords)
LABEL_X   = 4.75         # x position of annotation text (data coords)
GAP_FRAC  = 0.04         # gap between arrow start and y_mid, as fraction of y_max
SMOOTH_N  = 300          # points for smooth interpolation

# ── Smooth interpolation helper ───────────────────────────────────────────────
def smooth(x, y, n=SMOOTH_N):
    from scipy.interpolate import make_interp_spline
    x_new = np.linspace(x[0], x[-1], n)
    spl   = make_interp_spline(x, y, k=3)
    return x_new, spl(x_new)

# ── Draw each subplot ─────────────────────────────────────────────────────────
for ax, p in zip(axes, plots):
    gap = p["y_max"] * GAP_FRAC

    # Smooth curves
    ex_up, ey_up = smooth(epsilon, p["upper"])
    ex_dn, ey_dn = smooth(epsilon, p["lower"])

    # Filled area between curves (only if both are shown)
    if p["show_upper"] and p["show_lower"]:
        ax.fill_between(ex_up, ey_up, ey_dn, color=p["color"], alpha=p["alpha"])

    # Lines + markers (conditional on show_upper/show_lower)
    if p["show_upper"]:
        ax.plot(ex_up, ey_up, color=p["color"], linewidth=2.5, label="Global", zorder=10)
        ax.plot(epsilon, p["upper"], "o", color=p["color"], markersize=4, zorder=11)
    
    if p["show_lower"]:
        ax.plot(ex_dn, ey_dn, color=p["color"], linewidth=2.5, label="Global", zorder=10)
        ax.plot(epsilon, p["lower"], "o", color=p["color"], markersize=4, zorder=11)

    # Plot regional decomposition lines
    if "regions" in p and p["regions"]:
        for region in p["regions"]:
            ex_reg, ey_reg = smooth(epsilon, region["values"])
            ax.plot(ex_reg, ey_reg, color=region["color"], linewidth=1.2, 
                   linestyle=region["linestyle"], alpha=region["alpha"], 
                   label=region["name"], zorder=5)

    # Axes formatting
    ax.set_xlim(0, 8)
    ax.set_ylim(0, p["y_max"])
    ax.set_xlabel("ε in %", fontsize=11)
    ax.set_ylabel(f"HBI trade/production {p["y_label_add"]} in Mt", fontsize=10)
    ax.set_title(p["title"], fontsize=13, fontweight="medium", pad=10)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
    ax.set_xticks([0, 2, 4, 6, 8])
    
    # Add legend for regional lines
    if "regions" in p and p["regions"]:
        ax.legend(loc="best", fontsize=7, framealpha=0.9)

    # ── Arrows (data coords, drawn in axis transform) ──────────────────────────
    y_up_end = p["upper"][-1]
    y_dn_end = p["lower"][-1]
    y_up_start = p["y_mid"] + gap
    y_dn_start = p["y_mid"] - gap

    arrowprops = dict(
        arrowstyle="->",
        color="#444441",
        lw=1.4,
        mutation_scale=12,
    )

    # Upper arrow (pointing up toward curve) - only if arrow_up is defined
    if p["arrow_up"] is not None:
        ax.annotate(
            "", xy=(ARROW_X, y_up_end), xytext=(ARROW_X, y_up_start),
            xycoords="data", textcoords="data",
            arrowprops=arrowprops,
            clip_on=False,
        )

    # Lower arrow (pointing down toward curve) - only if arrow_dn is defined
    if p["arrow_dn"] is not None:
        ax.annotate(
            "", xy=(ARROW_X, y_dn_end), xytext=(ARROW_X, y_dn_start),
            xycoords="data", textcoords="data",
            arrowprops=arrowprops,
            clip_on=False,
        )

    # ── Arrow labels ──────────────────────────────────────────────────────────
    y_label_up = (y_up_start + y_up_end) / 2
    y_label_dn = (y_dn_start + y_dn_end) / 2

    if p["arrow_up"] is not None:
        ax.text(
            LABEL_X, y_label_up,
            "\n".join(p["arrow_up"]),
            fontsize=8, va="center", ha="left",
            color="#444441",
            clip_on=False,
        )
    
    if p["arrow_dn"] is not None:
        ax.text(
            LABEL_X, y_label_dn,
            "\n".join(p["arrow_dn"]),
            fontsize=8, va="center", ha="left",
            color="#444441",
            clip_on=False,
        )

# plt.savefig("mga_plots.png", dpi=150, bbox_inches="tight")
plt.show()
# print("Saved to mga_plots.png")

# MGA Analysis: HBI Production by Political Stability

In [ ]:
import os
import glob
import pandas as pd
import pypsa
from pypsa import Network

# ── Load snakemake and configuration ──────────────────────────────────────────

# Access MGA scenario configuration from snakemake.config
mga_config = snakemake.config.get("scenario", {}).get("mga-new-indicators-multiple", {})
mga_modifiers = mga_config.get("modifiers", {})
mga_slack_values = mga_modifiers.get("mga", {}).get("slack", [])
cost_year = "2050"
interone = "hbi"
intertwo = "eaf-grid"
final = "steel"
scenario = "mga-new-indicators-multiple"

print(f"MGA Configuration: {mga_config}")
print(f"MGA Modifiers: {mga_modifiers}")
print(f"Slack values from config: {mga_slack_values}")

In [ ]:

# ── Load PyPSA networks using NetworkCollection ───────────────────────────────

# Construct path to results directory
results_dir = os.path.abspath(os.path.join(
    os.path.dirname(snakemake.input.network),
    "..",
    scenario
))

if not os.path.exists(results_dir):
    results_dir = os.path.abspath(os.path.join(
        os.path.dirname(snakemake.input.network)
    ))

print(f"Results directory: {results_dir}")

# Load networks into a NetworkCollection
networks_dict = {}
slack_values = [0] + mga_slack_values

print(f"\nLoading networks for slack values: {slack_values}")

for slack in slack_values:
    if slack == 0:
        filepath = os.path.join(results_dir, "network_nan.nc")
        slack_key = None
    else:
        filepath = os.path.join(results_dir, f"network_{slack}.nc")
        slack_key = slack
    
    if os.path.exists(filepath):
        try:
            networks_dict[slack_key] = pypsa.Network(filepath)
            label = "optimal solution (slack=0)" if slack == 0 else f"slack={slack}"
            print(f"  ✓ Loaded {label}")
        except Exception as e:
            print(f"  ✗ Error loading {filepath}: {e}")
    else:
        print(f"  ✗ File not found: {filepath}")

# Create NetworkCollection from the loaded networks
# (Store as dict for now, can use NetworkCollection if available in this PyPSA version)
networks = networks_dict

print(f"\n✓ Successfully loaded {len(networks_dict)} networks")
print(f"✓ Network keys: {sorted(networks_dict.keys(), key=lambda x: (x is None, x))}")

In [ ]:
# ── Prepare MGA criteria: Political Stability ─────────────────────────────────

stability_file = snakemake.input.political_stability

print(f"Loading political stability from: {stability_file}")
stability_df = pd.read_csv(stability_file, index_col=0)
print("Political stability data:")
print(stability_df)

# Define threshold from MGA config
threshold_stability = 30

# Separate stable and unstable regions
stable_regions = stability_df[stability_df.iloc[:, 0] >= threshold_stability].index.tolist()
unstable_regions = stability_df[stability_df.iloc[:, 0] < threshold_stability].index.tolist()

print(f"\nStable regions (stability >= {threshold_stability}): {stable_regions}")
print(f"Unstable regions (stability < {threshold_stability}): {unstable_regions}")

all_regions = stable_regions + unstable_regions
print(f"\nTotal regions: {len(all_regions)}")
for region in sorted(all_regions):
    is_stable = region in stable_regions
    print(f"  {region}: {'STABLE' if is_stable else 'UNSTABLE'}")

In [ ]:
# ── Read PyPSA networks and extract HBI production data ───────────────────────

# Sort network keys: None (optimal) first, then MGA variants in ascending order
sorted_keys = sorted(networks_dict.keys(), key=lambda x: (x is not None, x))

# Create epsilon array for columns (0 for optimal, then slack values)
epsilon_values = [0 if k is None else k for k in sorted_keys]

# Initialize storage for regional HBI production
hbi_production_data = {region: [] for region in all_regions}

# Extract HBI production from each network
for net_key in sorted_keys:
    n = networks_dict[net_key]
    
    # Initialize regional totals for this network
    regional_totals = {region: 0 for region in all_regions}
    
    # Get HBI production from links (input side: bus0, p0)
    # Note: HBI links have ore as bus0 and HBI as bus1, with efficiency factor
    # The actual production flows through p0 (input side of the conversion)
    links_df = n.links
    
    for link_idx, link_row in links_df.iterrows():
        if link_row['carrier'] == 'hbi':
            bus1 = link_row['bus1']  # Output bus (e.g., "Europe_hbi")
            
            # Get the HBI production from this link
            # Use p0 (input ore amount) to track production
            if hasattr(n, 'links_t') and 'p0' in n.links_t:
                p_sum = n.links_t['p0'][link_idx].sum()
            else:
                p_sum = link_row.get('p_nom_opt', 0)
            
            if not pd.isna(p_sum) and p_sum > 0:
                # Determine which region this HBI output belongs to
                # Extract region from bus1 (e.g., "Europe_hbi" -> "Europe")
                found = False
                for region in all_regions:
                    if bus1.startswith(region):
                        regional_totals[region] += p_sum
                        found = True
                        break
            
            
    
    # Append regional totals for this network
    for region in all_regions:
        hbi_production_data[region].append(regional_totals[region])



# Create pandas DataFrame with regions as rows and slack values as columns
# Convert from tonnes to Mt (divide by 1 million)
df_hbi = pd.DataFrame(hbi_production_data, index=epsilon_values).T / 1e6

# Adjust from input (iron ore) to output (HBI)
df_hbi = df_hbi / snakemake.config["iron_ore"]["ore_to_steel_ratio"]

# Rename index to be more descriptive
df_hbi.index.name = 'Region'
df_hbi.columns.name = 'MGA Slack (ε)'

print("HBI Production by Region and MGA Slack (in Mt):")
df_hbi.round(1)

In [ ]:
# ── Append MGA criteria (political stability) to the dataframe ────────────────

# Add political stability column
df_hbi['Political Stability'] = stability_df.iloc[:, 0]

# Add stability classification (Stable/Unstable)
df_hbi['Stability Class'] = df_hbi.index.map(
    lambda region: 'Stable' if region in stable_regions else 'Unstable'
)

print("HBI Production with Political Stability:")
print(df_hbi)
print(f"\nStable regions: {len(stable_regions)}")
print(f"Unstable regions: {len(unstable_regions)}")

# ── Plot HBI production data ─────────────────────────────────────────────────

# Extract epsilon values and production data
epsilon_cols = df_hbi.columns[:-2]  # All columns except 'Political Stability' and 'Stability Class'
epsilon_vals = np.array([float(x) for x in epsilon_cols])

# Aggregate by stability class
total_stable_mt = df_hbi[df_hbi['Stability Class'] == 'Stable'][epsilon_cols].sum()
total_unstable_mt = df_hbi[df_hbi['Stability Class'] == 'Unstable'][epsilon_cols].sum()

# Convert to numpy arrays
total_stable_mt = np.array(total_stable_mt.values, dtype=float)
total_unstable_mt = np.array(total_unstable_mt.values, dtype=float)

# Get individual regional data
stable_production_mt = {region: np.array(df_hbi.loc[region, epsilon_cols].values, dtype=float)
                        for region in stable_regions if region in df_hbi.index}
unstable_production_mt = {region: np.array(df_hbi.loc[region, epsilon_cols].values, dtype=float)
                          for region in unstable_regions if region in df_hbi.index}

# Load region colors from snakemake config (if available)
tech_colors_file = snakemake.input.tech_colors if hasattr(snakemake.input, 'tech_colors') else None
region_colors = {}

if tech_colors_file and os.path.exists(tech_colors_file):
    import yaml
    with open(tech_colors_file, 'r') as f:
        config_colors = yaml.safe_load(f)
        # Extract region colors from config (usually under 'regions' key)
        if 'regions' in config_colors:
            region_colors = config_colors['regions']
        else:
            region_colors = config_colors

# Fallback colors if config not available
if not region_colors:
    region_colors = {
        # Stable regions
        'Central_America': '#1f77b4',
        'East_Asia': '#ff7f0e',
        'East_East_Asia': '#2ca02c',
        'Europe': '#d62728',
        'Far_West_Europe': '#9467bd',
        'Middle_East': '#8c564b',
        'North_America': '#e377c2',
        'Oceania': '#7f7f7f',
        'Pacific_Asia': '#17becf',
        'South_America': '#d62728',
        'South_South_America': '#aec7e8',
        'Other': '#1f77b4',
        # Unstable regions
        'Eurasia': '#ff7f0e',
        'North_West_Africa': '#2ca02c',
        'Subsaharan_Africa': '#9467bd',
        'West_Asia': '#8c564b',
    }

# Create single figure
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Settings
GAP_FRAC = 0.04

# ── Plot global (aggregated) line for stable regions (thicker, more prominent) ────
stable_color = "#3B6D11"
ax.plot(epsilon_vals, total_stable_mt, color=stable_color, linewidth=3, marker='o', 
        markersize=8, label="Global (Stable)", zorder=20)

# ── Plot all individual regional lines ────────────────────────────────────────

# Stable regions (solid lines)
for region in stable_regions:
    if region in stable_production_mt:
        region_data = stable_production_mt[region]
        if region_data.sum() > 0:  # Only plot if region has any production
            color = region_colors.get(region, '#000000')
            ax.plot(epsilon_vals, region_data, color=color, linewidth=1.2, 
                   linestyle='-', alpha=0.6, label=region, zorder=5)

# Unstable regions (dashed lines)
for region in unstable_regions:
    if region in unstable_production_mt:
        region_data = unstable_production_mt[region]
        if region_data.sum() > 0:  # Only plot if region has any production
            color = region_colors.get(region, '#000000')
            ax.plot(epsilon_vals, region_data, color=color, linewidth=1.2, 
                   linestyle='--', alpha=0.7, label=region, zorder=5)

# ── Axes formatting ──────────────────────────────────────────────────────────
ax.set_xlim(min(epsilon_vals) - 0.002, max(epsilon_vals) + 0.01)
ax.set_ylim(0, max(total_stable_mt) * 1.15)
ax.set_xlabel("MGA slack (fraction, ε)", fontsize=12)
ax.set_ylabel("HBI production in Mt", fontsize=12)
ax.set_title("HBI Production by Exporter Stability (All 16 Regions)", fontsize=14, fontweight="bold", pad=12)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
ax.legend(loc="best", fontsize=7, framealpha=0.95, ncol=3, title="Regions")

plt.tight_layout()
plt.show()

# ── Statistics ──────────────────────────────────────────────────────────────
print("✓ Plot with all 16 regions created successfully!")
print(f"\nStable regions HBI production range: {total_stable_mt.min():.4f} to {total_stable_mt.max():.4f} Mt")
if total_stable_mt[0] > 0:
    print(f"  Change: {((total_stable_mt[-1] / total_stable_mt[0]) - 1) * 100:.1f}%")
print(f"\nUnstable regions HBI production range: {total_unstable_mt.min():.4f} to {total_unstable_mt.max():.4f} Mt")
if total_unstable_mt[0] > 0:
    print(f"  Change: {((total_unstable_mt[-1] / total_unstable_mt[0]) - 1) * 100:.1f}%")

In [ ]:
# ── Append MGA criteria (political stability) to the dataframe ────────────────

# Add political stability column
df_hbi['Political Stability'] = stability_df.iloc[:, 0]

# Add stability classification (Stable/Unstable)
df_hbi['Stability Class'] = df_hbi.index.map(
    lambda region: 'Stable' if region in stable_regions else 'Unstable'
)

print("HBI Production with Political Stability:")

print(f"\nStable regions: {len(stable_regions)}")
print(f"Unstable regions: {len(unstable_regions)}")
df_hbi

# Misc

In [ ]:
n.statistics.energy_balance().div(1e6)